In [7]:
import json
from pdf2image import convert_from_path
import easyocr
import functions
import numpy as np
import importlib


importlib.reload(functions)


reader = easyocr.Reader(["ru", "en"], gpu=True)

file_path = "data/benchmark/Отсканированный документ.pdf"
preproc_path, _ = functions.preprocess_for_ocr(file_path)
pages = convert_from_path(preproc_path, dpi=200)

results = [reader.readtext(np.array(page), detail=1) for page in pages]
results = {
    "status": "success",
    "data": [
        {
            "page": i,
            "items": [
                {
                    "box": [
                        [
                            int(coord) for coord in point
                        ] for point in el[0]
                    ],
                    "text": el[1],
                    "confidence": el[2]
                } for el in page
            ]
        } for i, page in enumerate(results)
    ]
}

json_path = f"data/benchmark/predictions/easy_{preproc_path.split('/')[-1].removesuffix('.pdf')}.json"

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=4)

print(f"Файл успешно сохранён: {json_path}")

Предобработка завершена -> data/processed/proc_Отсканированный документ.pdf
Файл успешно сохранён: data/benchmark/predictions/easy_proc_Отсканированный документ.json


In [1]:
from doctr.io import DocumentFile
from doctr.models import ocr_predictor
import functions
import importlib
import json

importlib.reload(functions)


def format_doctr_to_json(result):
    formatted_data = []
    export_output = result.export()

    for i, page in enumerate(export_output["pages"]):
        page_height = page["dimensions"][0]
        page_width = page["dimensions"][1]

        page_items = []

        for block in page["blocks"]:
            for line in block["lines"]:
                for word in line["words"]:
                    (xmin, ymin), (xmax, ymax) = word["geometry"]

                    left = int(xmin * page_width)
                    top = int(ymin * page_height)
                    right = int(xmax * page_width)
                    bottom = int(ymax * page_height)

                    box = [
                        [left, top],
                        [right, top],
                        [right, bottom],
                        [left, bottom]
                    ]

                    page_items.append({
                        "box": box,
                        "text": word["value"],
                        "confidence": float(word["confidence"])
                    })

        formatted_data.append({
            "page": i + 1,
            "items": page_items
        })

    return {
        "status": "success",
        "data": formatted_data
    }


model = ocr_predictor(det_arch="db_resnet50", reco_arch="crnn_vgg16_bn", pretrained=True)

file_path = "data/benchmark/Отсканированный документ.pdf"
preproc_path, _ = functions.preprocess_for_ocr(file_path)
doc = DocumentFile.from_pdf(preproc_path)

result = model(doc)
final_results = format_doctr_to_json(result)

json_path = f"data/benchmark/predicts/doctr/{preproc_path.split('/')[-1].removesuffix('.pdf')}.json"

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(final_results, f, ensure_ascii=False, indent=4)

print(f"Файл успешно сохранён: {json_path}")

C:\My\PythonProjects\ML_edu\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Файл успешно сохранен: data/processed/proc_Отсканированный документ.pdf
Файл успешно сохранён: data/benchmark/predicts/doctr/proc_Отсканированный документ.json


In [2]:
from paddleocr import PaddleOCR
import paddle
import functions
import os


print("Compiled with CUDA:", paddle.is_compiled_with_cuda())
print("GPU count:", paddle.device.cuda.device_count())
print("Current device:", paddle.device.get_device())


ocr = PaddleOCR(lang="ru", use_gpu=True, use_angle_cls=True)


file_path = "data/docs/сканирование (2).pdf"
processed_path, _ = functions.preprocess_for_ocr(file_path)

temp_dir = "/data/temp"
os.makedirs(temp_dir, exist_ok=True)

try:
    if not processed_path.endswith((".jpg", ".png", ".pdf")):
        raise Exception("Неподдерживаемый формат файла. Используйте JPG, PNG или PDF.")

    results = ocr.ocr(processed_path)

    if not results:
        raise Exception("Текст на изображении не обнаружен")

    output = []

    for i, page_data in enumerate(results):
        page_items = []

        if page_data is None:
            output.append({"page": i + 1, "items": []})
            continue

        for line in page_data:
            box = line[0]
            text = line[1][0]
            score = line[1][1]

            page_items.append({
                "box": box,
                "text": text,
                "confidence": float(score)
            })

        output.append({
            "page": i + 1,
            "items": page_items
        })

    print(output)
except Exception as e:
    raise e

Запуск бенчмарка для http://localhost:8000/extract...
Количество итераций: 100
Итерация 1: 1.6477 сек
Итерация 2: 1.3989 сек
Итерация 3: 1.4280 сек
Итерация 4: 1.3517 сек
Итерация 5: 1.3519 сек
Итерация 6: 1.3598 сек
Итерация 7: 1.3504 сек
Итерация 8: 1.3435 сек
Итерация 9: 1.3576 сек
Итерация 10: 1.3737 сек
Итерация 11: 1.3452 сек
Итерация 12: 1.3430 сек
Итерация 13: 1.3919 сек
Итерация 14: 1.3559 сек
Итерация 15: 1.3314 сек
Итерация 16: 1.3707 сек
Итерация 17: 1.3456 сек
Итерация 18: 1.3813 сек
Итерация 19: 1.3607 сек
Итерация 20: 1.3607 сек
Итерация 21: 1.3507 сек
Итерация 22: 1.3814 сек
Итерация 23: 1.4374 сек
Итерация 24: 1.3555 сек
Итерация 25: 1.3679 сек
Итерация 26: 1.3756 сек
Итерация 27: 1.3472 сек
Итерация 28: 1.3496 сек
Итерация 29: 1.3494 сек
Итерация 30: 1.3600 сек
Итерация 31: 1.3518 сек
Итерация 32: 1.3797 сек
Итерация 33: 1.3437 сек
Итерация 34: 1.3490 сек
Итерация 35: 1.4058 сек
Итерация 36: 1.3524 сек
Итерация 37: 1.3554 сек
Итерация 38: 1.3641 сек
Итерация 39: 1.369

In [3]:
import requests
import time
import numpy as np


def benchmark_endpoint(url, file_path, iterations=100):
    print(f"Запуск бенчмарка для {url}...")
    print(f"Количество итераций: {iterations}")

    total_times = []

    for i in range(iterations):
        start_time = time.perf_counter()

        with open(file_path, "rb") as f:
            files = {"file": ("document.pdf", f, "application/pdf")}
            response = requests.post(url, files=files)

        end_time = time.perf_counter()

        if response.status_code == 200:
            duration = end_time - start_time
            total_times.append(duration)
            print(f"Итерация {i+1}: {duration:.4f} сек")
        else:
            print(f"Ошибка на итерации {i+1}")

    avg_time = np.mean(total_times)
    p95_time = np.percentile(total_times, 95)

    print(f"Среднее время: {avg_time:.4f} сек")
    print(f"P95: {p95_time:.4f} сек")
    print(f"Пропускная способность: {1/avg_time:.2f} док/сек")

_url = "http://localhost:8000/extract"
_file_path = "data/docs/сканирование (2).pdf"
benchmark_endpoint(_url, _file_path)

Обработка 1 одновременных запросов
Ресурсы: Cpu 20.5% | Gpu 6.0%
RPS: 0.57
Обработка 11 одновременных запросов
Ресурсы: Cpu 38.9% | Gpu 57.99999999999999%
RPS: 0.73
Обработка 21 одновременных запросов
Ресурсы: Cpu 48.4% | Gpu 1.0%
RPS: 0.74
Обработка 31 одновременных запросов
Ресурсы: Cpu 20.5% | Gpu 22.0%
RPS: 0.73
Обработка 41 одновременных запросов
Ресурсы: Cpu 9.8% | Gpu 86.0%
RPS: 0.73
Обработка 51 одновременных запросов
Ресурсы: Cpu 9.7% | Gpu 72.0%
RPS: 0.85
Доступность упала ниже 95%


In [ ]:
import asyncio
import httpx
import time
import psutil
import os
import GPUtil


def get_resource_usage():
    cpu_usage = psutil.cpu_percent(interval=0.1)
    gpu_usage = GPUtil.getGPUs()[0].load * 100
    return cpu_usage, gpu_usage


async def stress_test(url, file_path):
    loop = asyncio.get_running_loop()

    semaphore = asyncio.Semaphore(100)

    async def boxed_post(client, url, f_path):
        async with semaphore:
            with open(f_path, "rb") as f:
                try:
                    return await client.post(url, files={"file": f}, timeout=60.0)
                except Exception as e:
                    return e

    async with httpx.AsyncClient() as client:
        for i in range(1, 100, 10):
            tasks = [boxed_post(client, url, file_path) for _ in range(i)]
            tasks.append(loop.run_in_executor(None, get_resource_usage))

            start = time.perf_counter()
            results = await asyncio.gather(*tasks, return_exceptions=True)
            end = time.perf_counter()

            res_usage = results.pop(-1)

            if isinstance(res_usage, Exception):
                cpu, gpu = 0, 0
            else:
                cpu, gpu = res_usage

            failed_count = 0

            for res in results:
                if isinstance(res, Exception) or (hasattr(res, "status_code") and res.status_code != 200):
                    failed_count += 1

            _available = 1 - (failed_count / i)

            print(f"Обработка {i} одновременных запросов")
            print(f"Ресурсы: Cpu {cpu}% | Gpu {gpu}%")
            print(f"RPS: {i / (end - start):.2f}")

            if _available < 0.95:
                print(f"Доступность упала ниже 95%")
                break


_url = "http://localhost:8000/extract"
_file_path = "data/docs/сканирование (2).pdf"
await stress_test(_url, _file_path)

In [4]:
import base64
import requests
import io
from pdf2image import convert_from_path
import functions


def pdf_to_base64_images(pdf_path):
    images = convert_from_path(pdf_path, dpi=200)

    base64_strings = []

    for img in images:
        buffered = io.BytesIO()
        img.save(buffered, format="PNG")
        img_str = base64.b64encode(buffered.getvalue()).decode("utf-8")
        base64_strings.append(img_str)

    return base64_strings


url = "http://0.0.0.0:8000/v1/chat/completions"
file_path = "data/benchmark/Отсканированный документ.pdf"
processed_path, _ = functions.preprocess_for_ocr(file_path)

try:
    pages_base64 = pdf_to_base64_images(processed_path)
except Exception as e:
    print(f"Ошибка при конвертации PDF: {e}")
    pages_base64 = []

if pages_base64:
    content = [{"type": "text", "text": "Распознай текст со всех страниц документа:"}]

    for b64 in pages_base64:
        content.append({
            "type": "image_url",
            "image_url": {"url": f"data:image/jpeg;base64,{b64}"}
        })

    payload = {
        "model": "PaddlePaddle/PaddleOCR-VL",
        "messages": [
            {
                "role": "user",
                "content": content
            }
        ],
        "temperature": 0,
        "max_completion_tokens": 2048
    }

    response = requests.post(url, json=payload)

    if response.status_code == 200:
        result = response.json()
        print("Данные получены:", result["choices"][0]["message"]["content"])

        json_path = f"data/benchmark/predicts/paddle_vllm/{preproc_path.split('/')[-1].removesuffix('.pdf')}.json"

        with open(json_path, "w", encoding="utf-8") as f:
            json.dump(results, f, ensure_ascii=False, indent=4)
    else:
        print("Ошибка сервера:", response.status_code, response.text)

Файл успешно сохранен: data/processed/proc_Отсканированный документ.pdf
Ошибка сервера: 400 {"error":{"message":"Cannot merge different batch sizes for modality='image'! Found: batch_sizes={'image_grid_thw': 4, 'pixel_values': 1}","type":"BadRequestError","param":null,"code":400}}
